In [1]:
import math
import os
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms,models
import torchvision.utils as vutils
from tqdm.auto import tqdm

In [2]:

@dataclass
class Config:
  data_root:str='./data'
  batch_size:int=64
  num_workers:int=4
  image_size:int=32
  latent_channels:int=4

  # training
  epochs_ae:int=5
  epochs_ldm:int=10
  lr_ae:float=2e-4
  lr_disc:float=2e-4
  lr_ldm:float=2e-4

  # loss weights
  w_rec:float=1.0
  w_kl:float=1e-6
  w_lpips:float=0.5
  w_gan:float=0.1

  T:int=1000
  device:str='cuda' if torch.cuda.is_available() else 'cpu'
  save_dir:str='./checkpoints'

In [3]:
cfg = Config()
os.makedirs(cfg.save_dir,exist_ok=True)

In [4]:
transform = transforms.Compose([
    transforms.Resize(cfg.image_size),
    transforms.ToTensor(),
    # CIFAR10 is [0,1]; map to [-1,1]
    transforms.Normalize(mean=[0.5,0.5,0.5],std=[0.5,0.5,0.5]),
])

In [5]:

train_dataset = datasets.CIFAR10(root=cfg.data_root,train=True,download=True,transform=transform)
train_loader = DataLoader(train_dataset,batch_size=cfg.batch_size,shuffle=True,num_workers=cfg.num_workers,pin_memory=True)

100%|██████████| 170M/170M [29:34<00:00, 96.1kB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [6]:
# vae encoder & decoder

class VAEEncoder(nn.Module):
  def __init__(self,in_channels=3,latent_channels=4):
    super().__init__()
    ch = 64
    self.net = nn.Sequential(
        nn.Conv2d(in_channels,ch,3,stride=2,padding=1), # 32 -> 16
        nn.GroupNorm(32,ch),
        nn.SiLU(),
        nn.Conv2d(ch,ch*2,3,stride=2,padding=1), # 16-> 8
        nn.GroupNorm(32,ch*2),
        nn.SiLU(),
        nn.Conv2d(ch*2,ch*4,3,stride=1,padding=1),
        nn.GroupNorm(32,ch*4),
        nn.SiLU(),
        nn.Conv2d(ch*4,latent_channels*2,3,stride=1,padding=1)
    )

  def forward(self,x):
    h = self.net(x)
    mu,logvar = torch.chunk(h,2,dim=1)
    return mu,logvar

In [7]:
class VAEDecoder(nn.Module):
  def __init__(self,latent_channels=4,out_channels=3):
    super().__init__()
    ch = 64
    self.net = nn.Sequential(
        nn.Conv2d(latent_channels,ch*4,3,padding=1),
        nn.GroupNorm(32,ch*4),
        nn.SiLU(),
        nn.ConvTranspose2d(ch*4,ch*2,4,stride=2,padding=1), # 8 - 16
        nn.GroupNorm(32,ch*2),
        nn.SiLU(),
        nn.ConvTranspose2d(ch*2,ch,4,stride=2,padding=1), # 16 - 32
        nn.GroupNorm(32,ch),
        nn.SiLU(),
        nn.ConvTranspose2d(ch,out_channels,3,padding=1),
        nn.Tanh() # output in [-1,1]
    )

  def forward(self,z):
    return self.net(z)

In [8]:
class VAE(nn.Module):
  def __init__(self,in_channels=3,latent_channels=4):
    super().__init__()
    self.encoder = VAEEncoder(in_channels,latent_channels)
    self.decoder = VAEDecoder(latent_channels,in_channels)
    self.latent_channels = latent_channels

  def encode(self,x):
    mu,logvar = self.encoder(x)
    std = torch.exp(0.5*logvar)
    eps = torch.randn_like(std)
    z = mu+eps*std
    return z,mu,logvar

  def decode(self,z):
    return self.decoder(z)

  def forward(self,x):
    z,mu,logvar = self.encode(x)
    x_rec = self.decode(z)
    return x_rec,mu,logvar,z

In [9]:
class LPIPSPerceptualLoss(nn.Module):
  def __init__(self,device='cuda'):
    super().__init__()
    vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features

    self.layers = nn.ModuleDict({
        '0':vgg[:4],
        '1':vgg[4:9],
        '2':vgg[9:16],
    })

    self.layers.eval()
    for p in self.layers.parameters():
      p.requires_grad = False

    self.device = device
    self.layers.to(device)

    self.weights = nn.ParameterDict({
        '0':nn.Parameter(torch.ones(1)*0.5),
        '1':nn.Parameter(torch.ones(1)*0.5),
        '2':nn.Parameter(torch.ones(1)*0.5),
    })

    self.register_buffer('mean',torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
    self.register_buffer('std',torch.tensor([0.229,0.224,0.255]).view(1,3,1,1))

  def _normalize(self,x):
    # x in [-1,1] -> [0,1] -> ImageNet-norm
    x = (x+1.0)/2.0
    return (x-self.mean)/self.std

  def forward(self,x,x_rec):
    x = self._normalize(x)
    x_rec = self._normalize(x_rec)

    feats_x = {}
    feats_r = {}
    h = x
    hr = x_rec
    for name,layer in self.layers.items():
      h = layer(h)
      hr = layer(hr)
      feats_x[name] = h
      feats_r[name] = hr

    loss = 0.0
    for name in self.layers.keys():
      fx = feats_x[name]
      fr = feats_r[name]

      fx = F.normalize(fx.view(fx.size(0),-1),dim=1)
      fr = F.normalize(fr.view(fr.size(0),-1),dim=1)
      diff = (fx-fr).pow(2).mean(dim=1)
      loss = loss+self.weights[name]*diff.mean()

    return loss

In [10]:
class PatchGANDiscriminator(nn.Module):
  def __init__(self,in_channels=3):
    super().__init__()
    ch = 64
    def block(in_c,out_c,stride=2):
      return nn.Sequential(
          nn.Conv2d(in_c,out_c,4,stride=stride,padding=1),
          nn.LeakyReLU(0.2,inplace=True),
      )

    self.net = nn.Sequential(
        block(in_channels,ch,stride=2),
        block(ch,ch*2,stride=2),
        block(ch*2,ch*4,stride=2),
        block(ch*4,ch*8,stride=1),
        nn.Conv2d(ch*8,1,4,stride=1,padding=1)
    )

  def forward(self,x):
    return self.net(x)

In [11]:
def make_beta_schedule(T,schedule='cosine',beta_start=1e-4,beta_end=0.02):
  if schedule == 'linear':
    return torch.linspace(beta_start,beta_end,T)
  elif schedule=='cosine':
    steps = torch.arange(T+1,dtype=torch.float64)
    f = torch.cos(((steps/T)+0.008)/1.008*math.pi/2)**2
    alphas_bar = f/f[0]
    betas = 1 - (alphas_bar[1:]/alphas_bar[:-1])
    return betas.clamp(1e-5,0.999)
  else:
    raise ValueError('Unknown schedule')

In [12]:
class DiffusionSchedule:
  def __init__(self,T=1000,schedule='cosine',device='cuda'):
    self.device = device
    self.T = T
    betas = make_beta_schedule(T,schedule=schedule).to(device).to(torch.float32)
    alphas = 1.0-betas
    alphas_bar = torch.cumprod(alphas,dim=0)

    self.betas = betas
    self.alphas = alphas
    self.alphas_bar = alphas_bar

  def q_sample(self,x0,t,noise=None):
    if noise is None:
      noise = torch.randn_like(x0)
    alphas_bar_t = self.alphas_bar[t].view(-1,1,1,1)
    return torch.sqrt(alphas_bar_t)*x0+torch.sqrt(1.0-alphas_bar_t)*noise

In [13]:
class SinosoidalTimeEmbedding(nn.Module):
  def __init__(self,dim):
    super().__init__()
    self.dim = dim

  def forward(self,t):
    half_dim = self.dim//2
    freqs = torch.exp(
        torch.linspace(math.log(1.0),math.log(10000.0),half_dim,device=t.device)* -1.0
    )
    args = t.float().unsqueeze(1)*freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args),torch.cos(args)],dim=1)
    return emb

In [14]:
class TimeMLP(nn.Module):
  def __init__(self,time_dim,out_dim):
    super().__init__()
    self.mlp = nn.Sequential(
        nn.Linear(time_dim,out_dim),
        nn.SiLU(),
        nn.Linear(out_dim,out_dim)
    )

  def forward(self,t_emb):
    return self.mlp(t_emb)


In [15]:
class ResidualBlock(nn.Module):
  def __init__(self,in_channels,out_channels,time_channels):
    super().__init__()
    self.norm1 = nn.GroupNorm(32,in_channels)
    self.act1 = nn.SiLU()
    self.conv1 = nn.Conv2d(in_channels,out_channels,3,padding=1)

    self.norm2 = nn.GroupNorm(32,out_channels)
    self.act2 = nn.SiLU()
    self.conv2 = nn.Conv2d(out_channels,out_channels,3,padding=1)

    self.time_proj = nn.Linear(time_channels,out_channels)

    if in_channels != out_channels:
      self.skip = nn.Conv2d(in_channels,out_channels,1)
    else:
      self.skip = nn.Identity()

  def forward(self,x,t_emb):
    h = self.conv1(self.act1(self.norm1(x)))
    t_out = self.time_proj(t_emb).view(x.size(0),-1,1,1)
    h = h + t_out
    h = self.conv2(self.act2(self.norm2(h)))
    return h+self.skip(x)

In [16]:
class DownsampleBlock(nn.Module):
  def __init__(self,in_channels,out_channels,time_channels):
    super().__init__()
    self.res1 = ResidualBlock(in_channels,out_channels,time_channels)
    self.res2 = ResidualBlock(out_channels,out_channels,time_channels)
    self.down = nn.Conv2d(out_channels,out_channels,3,stride=2,padding=1)

  def forward(self,x,t_emb):
    x = self.res1(x,t_emb)
    x = self.res2(x,t_emb)
    skip = x
    x = self.down(x)
    return x,skip

In [17]:
class UpsampleBlock(nn.Module):
  def __init__(self,in_channels, skip_channels, out_channels,time_channels):
    super().__init__()
    # Upsample the main path first to match skip connection resolution
    self.up = nn.ConvTranspose2d(in_channels,in_channels,4,stride=2,padding=1)
    # Input to res1 will be x (upsampled in_channels) concatenated with skip (skip_channels)
    self.res1 = ResidualBlock(in_channels+skip_channels,out_channels,time_channels)
    self.res2 = ResidualBlock(out_channels,out_channels,time_channels)

  def forward(self,x,skip,t_emb):
    x = self.up(x) # Upsample x to match skip's spatial dimensions
    x = torch.cat([x,skip],dim=1)
    x = self.res1(x,t_emb)
    x = self.res2(x,t_emb)
    return x

In [18]:
class LatentUNet(nn.Module):
  def __init__(self,latent_channels=4,base_channels=64,time_emb=256):
    super().__init__()
    self.time_embed = SinosoidalTimeEmbedding(time_emb)
    self.time_mlp = TimeMLP(time_emb,time_emb)

    self.in_conv = nn.Conv2d(latent_channels,base_channels,3,padding=1)

    self.down1 = DownsampleBlock(base_channels,base_channels,time_emb)
    self.down2 = DownsampleBlock(base_channels,base_channels*2,time_emb)

    self.bot1 = ResidualBlock(base_channels*2,base_channels*2,time_emb)
    self.bot2 = ResidualBlock(base_channels*2,base_channels*2,time_emb)

    # Updated UpsampleBlock instantiations with skip_channels
    self.up2 = UpsampleBlock(base_channels*2, base_channels*2, base_channels, time_emb)
    self.up1 = UpsampleBlock(base_channels, base_channels, base_channels, time_emb)

    self.out_norm = nn.GroupNorm(32,base_channels)
    self.out_act = nn.SiLU()
    self.out_conv = nn.Conv2d(base_channels,latent_channels,3,padding=1)

  def forward(self,z_t,t):
    t_emb = self.time_mlp(self.time_embed(t))
    x = self.in_conv(z_t)

    x,skip1 = self.down1(x,t_emb)
    x,skip2 = self.down2(x,t_emb)

    x = self.bot1(x,t_emb)
    x = self.bot2(x,t_emb)

    x = self.up2(x,skip2,t_emb)
    x = self.up1(x,skip1,t_emb)

    x = self.out_conv(self.out_act(self.out_norm(x)))
    return x

In [19]:
def train_vae_with_lpips_gan(vae,disc,lpips_loss,train_loader,cfg):
  device = cfg.device
  vae.to(device)
  disc.to(device)
  lpips_loss.to(device)

  opt_vae = torch.optim.AdamW(vae.parameters(),lr=cfg.lr_ae)
  opt_disc = torch.optim.AdamW(disc.parameters(),lr=cfg.lr_disc)
  bce_with_logits = nn.BCEWithLogitsLoss()

  vae.train()
  disc.train()

  for epoch in range(cfg.epochs_ae):
    running_loss_vae = 0.0
    running_loss_disc = 0.0

    # Wrap train_loader with tqdm for a progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs_ae} [VAE+GAN]")
    for x, _ in pbar:
      x = x.to(device)
      B = x.size(0)


      # train discriminator

      with torch.no_grad():
        x_rec,_,_,_ = vae(x)

      # real images; label = 1
      logits_real = disc(x)
      ones = torch.ones_like(logits_real)
      loss_disc_real = bce_with_logits(logits_real,ones)

      # fake images label =0
      logits_fake = disc(x_rec.detach())
      zeros = torch.zeros_like(logits_fake)
      loss_disc_fake = bce_with_logits(logits_fake,zeros)

      loss_disc = (loss_disc_real+loss_disc_fake)/2
      opt_disc.zero_grad()
      loss_disc.backward()
      opt_disc.step()

      running_loss_disc += loss_disc.item() *B

      # train vae

      x_rec,mu,logvar,z = vae(x)

      # reconstruction loss
      loss_rec_l1 = F.l1_loss(x_rec,x)
      loss_rec_l2 = F.mse_loss(x_rec,x)
      loss_rec = cfg.w_rec*(loss_rec_l1+loss_rec_l2)

      # kl loss
      kl_loss = -0.5*torch.mean(1+logvar-mu.pow(2)-logvar.exp())
      loss_kl = cfg.w_kl*kl_loss

      # lpips loss
      loss_lpips = cfg.w_lpips*lpips_loss(x,x_rec)

      # adversarial loss for vae as generator
      logits_fake_g = disc(x_rec)
      ones_g = torch.ones_like(logits_fake_g)
      loss_gan = cfg.w_gan*bce_with_logits(logits_fake_g,ones_g)

      loss_vae = loss_rec+loss_kl+loss_lpips+loss_gan

      opt_vae.zero_grad()
      loss_vae.backward()
      opt_vae.step()

      running_loss_vae += loss_vae.item()*B

    epoch_loss_vae = running_loss_vae/len(train_loader.dataset)
    epoch_loss_disc = running_loss_disc/len(train_loader.dataset)

    print(
        f"[VAE+GAN] Epoch {epoch+1}/{cfg.epochs_ae}- "
        f"VAE LOSS: {epoch_loss_vae:.4f},Disc Loss: {epoch_loss_disc:.4f}"
    )

    torch.save(vae.state_dict(),os.path.join(cfg.save_dir,'vae_first_stage.pt'))
    torch.save(disc.state_dict(),os.path.join(cfg.save_dir,'disc_first_stage.pt'))

In [20]:
def train_latent_diffusion(vae,unet,train_loader,cfg):
  device = cfg.device
  schedule = DiffusionSchedule(T=cfg.T,schedule="cosine",device=device)

  vae.to(device)
  vae.eval()

  for p in vae.parameters():
    p.requires_grad = False

  unet.to(device)
  optimizer = torch.optim.AdamW(unet.parameters(),lr=cfg.lr_ldm)

  unet.train()
  for epoch in range(cfg.epochs_ldm):
    running_loss = 0.0
    # Wrap train_loader with tqdm for a progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs_ldm} [LDM]")
    for x,_ in pbar:
      x = x.to(device)

      with torch.no_grad():
        # Corrected unpacking: vae.encode returns 3 values (z, mu, logvar)
        z, _, _ = vae.encode(x)

      B = z.size(0)

      t = torch.randint(1,cfg.T,(B,),device=device)
      noise = torch.randn_like(z)

      z_t = schedule.q_sample(z,t,noise=noise)
      eps_pred = unet(z_t,t)

      loss = F.mse_loss(eps_pred,noise)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      running_loss += loss.item()*x.size(0)

    epoch_loss = running_loss/len(train_loader.dataset)
    print(f"[LDM] Epoch {epoch+1}/{cfg.epochs_ldm}- Loss: {epoch_loss:.4f}")
    torch.save(unet.state_dict(),os.path.join(cfg.save_dir,'latent_unet.pt'))

In [21]:
@torch.no_grad()
def p_sample_step(schedule,unet,z_t,t):
  betas = schedule.betas
  alphas = schedule.alphas
  alphas_bar = schedule.alphas_bar

  beta_t = betas[t].view(-1,1,1,1)
  alpha_t = alphas[t].view(-1,1,1,1)
  alpha_bar_t = alphas_bar[t].view(-1,1,1,1)

  eps_theta = unet(z_t,t)

  coef1 = 1.0/torch.sqrt(alpha_t)
  coef2 = beta_t/torch.sqrt(1.0-alpha_bar_t)
  mean = coef1*(z_t-coef2*eps_theta)

  if t[0].item()>0:
    sigma_t = torch.sqrt(beta_t)
    noise = torch.randn_like(z_t)
    z_prev = mean + sigma_t*noise
  else:
    z_prev = mean

  return z_prev

@torch.no_grad()
def sample_images(vae,unet,cfg,num_samples=16):
  device = cfg.device
  schedule = DiffusionSchedule(T=cfg.T,schedule="cosine",device=device)

  vae.to(device)
  unet.to(device)
  vae.eval()
  unet.eval()

  z_t = torch.randn(num_samples,cfg.latent_channels,8,8,device=device)

  for t_int in reversed(range(cfg.T)):
    t = torch.full((num_samples,),t_int,dtype=torch.long,device=device)
    z_t = p_sample_step(schedule,unet,z_t,t)

  x_gen = vae.decode(z_t)

  x_gen = (x_gen+1.0)/2.0
  x_gen = x_gen.clamp(0.0,1.0)

  return x_gen

def save_sample(x_gen,filename,nrow=4):
  grid = vutils.make_grid(x_gen,nrow=nrow,padding=2,pad_value=1.0)
  vutils.save_image(grid,filename)
  print(f"Saved image to {filename}")

In [22]:
def main():
    device = cfg.device
    print(f"Using device: {device}")

    vae = VAE(in_channels=3,latent_channels=cfg.latent_channels)
    disc = PatchGANDiscriminator(in_channels=3)
    lpips_loss = LPIPSPerceptualLoss(device=device)
    unet = LatentUNet(latent_channels=cfg.latent_channels,base_channels=64,time_emb=256)

    # Stage 1: Train VAE + LPIPS + GAN
    print("=== Training VAE with LPIPS + GAN ===")
    train_vae_with_lpips_gan(vae, disc, lpips_loss, train_loader, cfg)

    # Stage 2: Train latent diffusion
    print("=== Training Latent Diffusion Model ===")
    train_latent_diffusion(vae, unet, train_loader, cfg)

    # Sampling
    print("=== Sampling from Latent Diffusion Model ===")
    vae.load_state_dict(torch.load(os.path.join(cfg.save_dir, "vae_first_stage.pt"), weights_only=True))
    unet.load_state_dict(torch.load(os.path.join(cfg.save_dir, "latent_unet.pt"), weights_only=True))

    x_gen = sample_images(vae, unet, cfg, num_samples=16)
    save_sample(x_gen, os.path.join(cfg.save_dir, "ldm_vae_samples_cifar10.png"), nrow=4)

In [23]:
main()

Using device: cuda
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 185MB/s]


=== Training VAE with LPIPS + GAN ===


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 1/5 [VAE+GAN]:   0%|          | 0/782 [00:00<?, ?it/s]

...

[LDM] Epoch 8/10- Loss: 0.3380


Epoch 9/10 [LDM]:   0%|          | 0/782 [00:00<?, ?it/s]

[LDM] Epoch 9/10- Loss: 0.3347


Epoch 10/10 [LDM]:   0%|          | 0/782 [00:00<?, ?it/s]

[LDM] Epoch 10/10- Loss: 0.3345
=== Sampling from Latent Diffusion Model ===
Saved image to ./checkpoints/ldm_vae_samples_cifar10.png
